In [2]:
import torch
import pandas as pd
import os
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

In [3]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = fasterrcnn_resnet50_fpn(weights=None, num_classes=5)
model.load_state_dict(torch.load("best_model.pth"))
model = model.to(device)
model.eval()

print("Damage model loaded.")

C:\Users\HP\AppData\Local\Temp\ipykernel_5604\3081402135.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pth"))


Damage model loaded.


In [4]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [5]:
import numpy as np
from scipy.stats import entropy

def extract_features(pred, image_shape, score_threshold=0.5):

    h, w = image_shape
    image_area = h * w

    boxes = pred["boxes"].detach().cpu().numpy()
    labels = pred["labels"].detach().cpu().numpy()
    scores = pred["scores"].detach().cpu().numpy()

    valid = scores > score_threshold
    boxes = boxes[valid]
    labels = labels[valid]
    scores = scores[valid]

    num_detections = len(boxes)

    box_area_ratios = []
    class_counts = [0,0,0,0]

    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = box
        area = (x2-x1)*(y2-y1)
        ratio = area / image_area
        box_area_ratios.append(ratio)

        if label <= 4:
            class_counts[label-1] += 1

    total_damage_area_ratio = sum(box_area_ratios)

    avg_box_area_ratio = np.mean(box_area_ratios) if num_detections > 0 else 0
    max_box_area_ratio = np.max(box_area_ratios) if num_detections > 0 else 0
    min_box_area_ratio = np.min(box_area_ratios) if num_detections > 0 else 0
    var_box_area_ratio = np.var(box_area_ratios) if num_detections > 0 else 0

    damage_density = total_damage_area_ratio / num_detections if num_detections > 0 else 0

    class_prob = np.array(class_counts) / num_detections if num_detections > 0 else np.zeros(4)
    class_entropy = entropy(class_prob) if num_detections > 0 else 0

    mean_confidence = np.mean(scores) if len(scores)>0 else 0

    return [
        num_detections,
        total_damage_area_ratio,
        avg_box_area_ratio,
        max_box_area_ratio,
        min_box_area_ratio,
        var_box_area_ratio,
        damage_density,
        class_counts[0],
        class_counts[1],
        class_counts[2],
        class_counts[3],
        class_entropy,
        mean_confidence
    ]

In [6]:
def process_folder(folder_path, label):
    data = []

    for img_name in tqdm(os.listdir(folder_path)):
        img_path = os.path.join(folder_path, img_name)

        image = Image.open(img_path).convert("RGB")
        img_tensor = transform(image).to(device)

        with torch.no_grad():
            prediction = model([img_tensor])[0]

        features = extract_features(
            prediction,
            image_shape=image.size[::-1]
        )

        features.append(label)
        data.append(features)

    return data

In [7]:
TRAIN_FRAUD_DIR = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\data2\train\Fraud"
TRAIN_NONFRAUD_DIR = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\data2\train\Non-Fraud"

train_data = []

train_data += process_folder(TRAIN_FRAUD_DIR, label=1)
train_data += process_folder(TRAIN_NONFRAUD_DIR, label=0)

columns = [
    "num_detections",
    "total_damage_area_ratio",
    "avg_box_area_ratio",
    "max_box_area_ratio",
    "min_box_area_ratio",
    "var_box_area_ratio",
    "damage_density",
    "class_1_count",
    "class_2_count",
    "class_3_count",
    "class_4_count",
    "class_entropy",
    "mean_confidence",
    "label"
]

df_train = pd.DataFrame(train_data, columns=columns)
df_train.to_csv("fraud_train_features.csv", index=False)

print("Train features saved.")

100%|██████████| 6091/6091 [14:03<00:00,  7.22it/s]


Train features saved.


In [8]:
TEST_FRAUD_DIR = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\data2\test\Fraud"
TEST_NONFRAUD_DIR = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\data2\test\Non-Fraud"

test_data = []

test_data += process_folder(TEST_FRAUD_DIR, label=1)
test_data += process_folder(TEST_NONFRAUD_DIR, label=0)

columns = [
    "num_detections",
    "total_damage_area_ratio",
    "avg_box_area_ratio",
    "max_box_area_ratio",
    "min_box_area_ratio",
    "var_box_area_ratio",
    "damage_density",
    "class_1_count",
    "class_2_count",
    "class_3_count",
    "class_4_count",
    "class_entropy",
    "mean_confidence",
    "label"
]

df_test = pd.DataFrame(test_data, columns=columns)
df_test.to_csv("fraud_test_features.csv", index=False)

print("test features saved.")

100%|██████████| 1523/1523 [03:31<00:00,  7.19it/s]

test features saved.


In [9]:
print(df_train.shape)
print(df_train['label'].value_counts())

(6536, 14)
label
0    6091
1     445
Name: count, dtype: int64
